# Imports and Setup

# GNN vs MLP on the Cora Citation Network (Student Version)

In this notebook you will compare:

- A **Multilayer Perceptron (MLP)** that uses only node features.
- A **Graph Convolutional Network (GCN)** that uses node features **and** the citation graph.

Dataset: **Cora citation network**

- Each node is a paper.
- Edges are citation links.
- Features are word indicators.
- Labels are paper topics (7 classes).

You have two main TODOs:

1. Implement adjacency normalization for the GCN.
2. Implement the forward pass of a GCN layer.

Useful PyTorch documentation:

- Main docs: https://pytorch.org/docs/stable/index.html
- `torch.nn` layers: https://pytorch.org/docs/stable/nn.html
- Tensor operations: https://pytorch.org/docs/stable/tensors.html


In [1]:
import math
import random
import urllib.request

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

device = torch.device("mps" if torch.mps.is_available() else "cpu") # replace mps with cuda depending on OS and GPU
print("Using device:", device)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)


Using device: cpu


## 1. Download the Cora dataset

We will download two text files from a public GitHub mirror:

- `cora.content`: one line per paper  
  Format: `paper_id f1 f2 ... f1433 class_label`
- `cora.cites`: citation edges  
  Format: `cited_paper_id citing_paper_id`

We use the standard `urllib.request` module.


In [2]:
CONTENT_URL = "https://raw.githubusercontent.com/asolayman/cora_dataset/master/cora.content"
CITES_URL = "https://raw.githubusercontent.com/asolayman/cora_dataset/master/cora.cites"

def download_text(url: str) -> str:
    with urllib.request.urlopen(url) as f:
        return f.read().decode("utf-8")

content_text = download_text(CONTENT_URL)
cites_text = download_text(CITES_URL)

print("Sample from cora.content:")
print("\n".join(content_text.splitlines()[:2]))
print("\nSample from cora.cites:")
print("\n".join(cites_text.splitlines()[:2]))


Sample from cora.content:
31336	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	

## 2. Parse `cora.content` into feature matrix `X` and labels `y`

Steps:

1. Read each line of `cora.content`.
2. Extract:
   - `paper_id`
   - feature vector (1433 binary features)
   - class label (string)
3. Map string labels to integer class indices.
4. Build:
   - `X` of shape `(N, F)` as `torch.float32`
   - `y` of shape `(N,)` as `torch.long`

Related docs:

- Tensor creation ops:  
  https://pytorch.org/docs/stable/tensors.html#creation-ops
- Data types:  
  https://pytorch.org/docs/stable/tensors.html#data-types


In [3]:
paper_ids = []
features_list = []
labels_list = []

for line in content_text.splitlines():
    parts = line.strip().split()
    pid = parts[0]
    *feat_bits, label = parts[1:]
    paper_ids.append(pid)
    features_list.append([int(v) for v in feat_bits])
    labels_list.append(label)

paper_ids = np.array(paper_ids)
X_np = np.array(features_list, dtype=np.float32)
labels_str = np.array(labels_list)

# Encode labels (string -> integer)
classes = sorted(set(labels_str))
class_to_idx = {c: i for i, c in enumerate(classes)}
y_np = np.array([class_to_idx[c] for c in labels_str], dtype=np.int64)
num_classes = len(classes)

# Convert to tensors
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.long)

num_nodes, num_features = X.shape
print(f"Num nodes: {num_nodes}, Num features: {num_features}, Num classes: {num_classes}")
print("Classes:", classes)


Num nodes: 2708, Num features: 1433, Num classes: 7
Classes: [np.str_('Case_Based'), np.str_('Genetic_Algorithms'), np.str_('Neural_Networks'), np.str_('Probabilistic_Methods'), np.str_('Reinforcement_Learning'), np.str_('Rule_Learning'), np.str_('Theory')]


## 3. Build adjacency matrix `A` from `cora.cites`

We now use the citation file to build an adjacency matrix `A`:

- Create a mapping from `paper_id` to row index.
- For each citation `(u, v)`:
  - Add an undirected edge between the corresponding nodes.
  - Set `A[i, j] = A[j, i] = 1`.

We will later add self-loops and normalize this matrix for the GCN.


In [4]:
id_to_idx = {pid: i for i, pid in enumerate(paper_ids)}
N = len(paper_ids)

A_np = np.zeros((N, N), dtype=np.float32)
missing = 0

for line in cites_text.splitlines():
    parts = line.strip().split()
    if len(parts) != 2:
        continue
    u, v = parts
    if u in id_to_idx and v in id_to_idx:
        i, j = id_to_idx[u], id_to_idx[v]
        A_np[i, j] = 1.0
        A_np[j, i] = 1.0
    else:
        missing += 1

A = torch.tensor(A_np, dtype=torch.float32)

print("Adjacency shape:", A.shape)
print("Approx number of undirected edges:", int(A.sum().item() / 2))
print("Missing citation pairs:", missing)


Adjacency shape: torch.Size([2708, 2708])
Approx number of undirected edges: 5278
Missing citation pairs: 0


## 4. Train / validation / test split

We create three boolean masks:

- `train_mask`
- `val_mask`
- `test_mask`

We use scikit-learn's `train_test_split` with stratification on `y` so that the class distribution is preserved.

Suggested split:

- 60% training
- 20% validation
- 20% test

Scikit-learn docs:  
https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html


In [5]:
indices = np.arange(N)

idx_train, idx_temp, y_train, y_temp = train_test_split(
    indices, y_np, test_size=0.4, random_state=42, stratify=y_np
)
idx_val, idx_test, y_val, y_test = train_test_split(
    idx_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

train_mask = torch.zeros(N, dtype=torch.bool)
val_mask = torch.zeros(N, dtype=torch.bool)
test_mask = torch.zeros(N, dtype=torch.bool)

train_mask[idx_train] = True
val_mask[idx_val] = True
test_mask[idx_test] = True

print("Train nodes:", train_mask.sum().item())
print("Val nodes:  ", val_mask.sum().item())
print("Test nodes: ", test_mask.sum().item())


Train nodes: 1624
Val nodes:   542
Test nodes:  542


## 5. TODO 1 – Normalize the adjacency matrix for GCN

For the GCN we use the following normalization (Kipf & Welling, 2017):

1. Add self-loops:  
   $\tilde{A} = A + I$

2. Compute degree matrix \(D\):  
   $D_{ii} = \sum_j \tilde{A}_{ij}$

3. Compute $D^{-1/2}$.

4. Compute normalized adjacency:  
   $
   \hat{A} = D^{-1/2} \tilde{A} D^{-1/2}
   $

Your task:

Implement `normalize_adjacency(A)` using PyTorch operations only.

Helpful functions:

- `torch.eye`: https://pytorch.org/docs/stable/generated/torch.eye.html  
- `torch.pow`: https://pytorch.org/docs/stable/generated/torch.pow.html  
- `torch.isinf`: https://pytorch.org/docs/stable/generated/torch.isinf.html
- `@` for matrix multiplication: https://pytorch.org/docs/stable/generated/torch.matmul.html


In [6]:
def normalize_adjacency(A: torch.Tensor) -> torch.Tensor:
    """
    TODO 1:
    Implement symmetric normalization: A_hat = D^{-1/2} (A + I) D^{-1/2}

    A: (N, N) adjacency matrix without self-loops.

    Steps:
    1) Add self-loops: A_tilde = A + I
    2) Compute degree vector: deg = row-wise sum of A_tilde
    3) Compute deg_inv_sqrt = deg ** (-0.5), handle infinities
    4) Build diagonal matrix D_inv_sqrt
    5) Return A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt
    """


    A_tilde = A + torch.eye(A.size(0), device=A.device)
    degree = torch.sum(A_tilde, dim=1)
    for i in range(degree.size(0)):
        if degree[i] == 0:
            degree[i] = 1.0  # to avoid division by zero
    deg_inv_sqrt = torch.pow(degree, -0.5)
    D_inv_sqrt = torch.diag(deg_inv_sqrt)
    A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt
    
    return A_hat



## 6. Define models: MLP and GCN (with TODO 2)

We define two models:

1. `MLPClassifier` (baseline, no graph):
   - Input: node features `X`
   - Ignores adjacency `A`
   - Architecture: `Linear -> ReLU -> Linear`

2. `GCNLayer` (one GCN layer) – TODO 2:
   - Implements:
     $
     H' = \hat{A} H W
     $
     where:
     - $H$ is the input node feature matrix
     - $W$ is a learnable weight matrix
     - $\hat{A}$ is the normalized adjacency

3. `SimpleGCN`:
   - Two GCN layers:
     - `GCNLayer -> ReLU -> GCNLayer`

Your task in TODO 2:

Implement the `forward` of `GCNLayer` using:

- Neighbor aggregation: `H_agg = A_norm @ X`
- Linear transform: `out = H_agg @ self.weight`

Useful docs:

- `nn.Linear`: https://pytorch.org/docs/stable/generated/torch.nn.Linear.html  
- `nn.Parameter`: https://pytorch.org/docs/stable/generated/torch.nn.Parameter.html  
- `torch.nn.functional.relu`: https://pytorch.org/docs/stable/nn.functional.html#relu


In [8]:
class MLPClassifier(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = F.relu(self.fc1(x))
        out = self.fc2(h)
        return out


class GCNLayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_dim, out_dim) * 0.1)

    def forward(self, X: torch.Tensor, A_norm: torch.Tensor) -> torch.Tensor:
        """
        TODO 2:
        Implement one GCN layer.

        X: (N, F_in)
        A_norm: (N, N)

        Steps:
        1) Aggregate neighbors: H_agg = A_norm @ X
        2) Linear transform: out = H_agg @ self.weight
        3) Return out
        """

        H_agg = A_norm @ X
        out = H_agg @ self.weight
        return out



class SimpleGCN(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int):
        super().__init__()
        self.gcn1 = GCNLayer(in_dim, hidden_dim)
        self.gcn2 = GCNLayer(hidden_dim, out_dim)

    def forward(self, X: torch.Tensor, A_norm: torch.Tensor) -> torch.Tensor:
        h = self.gcn1(X, A_norm)
        h = F.relu(h)
        out = self.gcn2(h, A_norm)
        return out


## 7. Training utilities: accuracy and generic training loop

We define:

1. `accuracy(logits, y, mask)`:
   - Computes accuracy over the nodes where `mask` is `True`.

2. `train_model(...)`:
   - Works for both MLP and GCN:
     - If `A_norm` is `None`, calls `model(X)`.
     - If `A_norm` is not `None`, calls `model(X, A_norm)`.

Loss function: `cross_entropy` from `torch.nn.functional`.

Documentation:

- Cross entropy:  
  https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html  
- Optimizers:  
  https://pytorch.org/docs/stable/optim.html


In [9]:
def accuracy(logits: torch.Tensor, y: torch.Tensor, mask: torch.Tensor) -> float:
    preds = logits[mask].argmax(dim=-1)
    correct = (preds == y[mask]).float().mean()
    return correct.item()


def train_model(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    X: torch.Tensor,
    y: torch.Tensor,
    train_mask: torch.Tensor,
    val_mask: torch.Tensor,
    test_mask: torch.Tensor,
    A_norm: torch.Tensor = None,
    num_epochs: int = 200,
    verbose: bool = True,
):
    model.to(device)
    X = X.to(device)
    y = y.to(device)
    train_mask = train_mask.to(device)
    val_mask = val_mask.to(device)
    test_mask = test_mask.to(device)
    if A_norm is not None:
        A_norm = A_norm.to(device)

    for epoch in range(1, num_epochs + 1):
        model.train()
        optimizer.zero_grad()

        if A_norm is None:
            logits = model(X)
        else:
            logits = model(X, A_norm)

        loss = F.cross_entropy(logits[train_mask], y[train_mask])
        loss.backward()
        optimizer.step()

        if verbose and epoch % 40 == 0:
            model.eval()
            with torch.no_grad():
                if A_norm is None:
                    logits_eval = model(X)
                else:
                    logits_eval = model(X, A_norm)
            train_acc = accuracy(logits_eval, y, train_mask)
            val_acc = accuracy(logits_eval, y, val_mask)
            print(
                f"Epoch {epoch:03d} | Loss: {loss.item():.4f} | "
                f"Train acc: {train_acc:.3f} | Val acc: {val_acc:.3f}"
            )

    model.eval()
    with torch.no_grad():
        if A_norm is None:
            logits = model(X)
        else:
            logits = model(X, A_norm)

    train_acc = accuracy(logits, y, train_mask)
    val_acc = accuracy(logits, y, val_mask)
    test_acc = accuracy(logits, y, test_mask)

    return logits, train_acc, val_acc, test_acc


## 8. Train the MLP (ANN baseline)

In this step:

- We train an MLP that uses only the node features `X`.
- It ignores the graph structure (`A` is not used).

You can already run this cell **even before** completing TODO 1 and TODO 2.


In [10]:
in_dim = num_features
hidden_dim = 16

mlp = MLPClassifier(in_dim, hidden_dim, num_classes)
opt_mlp = torch.optim.Adam(mlp.parameters(), lr=0.01, weight_decay=5e-4)

print("=== Training MLP (baseline) ===")
logits_mlp, train_acc_mlp, val_acc_mlp, test_acc_mlp = train_model(
    mlp,
    opt_mlp,
    X,
    y,
    train_mask,
    val_mask,
    test_mask,
    A_norm=None,
    num_epochs=200,
    verbose=True,
)

print(f"\nMLP final train acc: {train_acc_mlp:.3f}")
print(f"MLP final val acc  : {val_acc_mlp:.3f}")
print(f"MLP final test acc : {test_acc_mlp:.3f}")


=== Training MLP (baseline) ===
Epoch 040 | Loss: 0.0784 | Train acc: 0.995 | Val acc: 0.727
Epoch 080 | Loss: 0.0474 | Train acc: 0.999 | Val acc: 0.738
Epoch 120 | Loss: 0.0354 | Train acc: 0.999 | Val acc: 0.740
Epoch 160 | Loss: 0.0291 | Train acc: 0.999 | Val acc: 0.736
Epoch 200 | Loss: 0.0252 | Train acc: 0.999 | Val acc: 0.731

MLP final train acc: 0.999
MLP final val acc  : 0.731
MLP final test acc : 0.744


## 9. Train the GCN (GNN)

Now we train the 2-layer GCN.

Important:

- You must first complete:
  - TODO 1: `normalize_adjacency`
  - TODO 2: `GCNLayer.forward`

Then:

1. Compute `A_norm = normalize_adjacency(A)`.
2. Train `SimpleGCN` using the same training loop.
3. Compare test accuracy with the MLP baseline.


In [11]:
# Make sure TODO 1 and TODO 2 are implemented before running this cell.

A_norm = normalize_adjacency(A)

gcn = SimpleGCN(in_dim, hidden_dim, num_classes)
opt_gcn = torch.optim.Adam(gcn.parameters(), lr=0.01, weight_decay=5e-4)

print("=== Training GCN (GNN) ===")
logits_gcn, train_acc_gcn, val_acc_gcn, test_acc_gcn = train_model(
    gcn,
    opt_gcn,
    X,
    y,
    train_mask,
    val_mask,
    test_mask,
    A_norm=A_norm,
    num_epochs=200,
    verbose=True,
)

print(f"\nGCN final train acc: {train_acc_gcn:.3f}")
print(f"GCN final val acc  : {val_acc_gcn:.3f}")
print(f"GCN final test acc : {test_acc_gcn:.3f}")


=== Training GCN (GNN) ===
Epoch 040 | Loss: 0.2289 | Train acc: 0.933 | Val acc: 0.893
Epoch 080 | Loss: 0.1330 | Train acc: 0.972 | Val acc: 0.878
Epoch 120 | Loss: 0.1011 | Train acc: 0.982 | Val acc: 0.880
Epoch 160 | Loss: 0.0844 | Train acc: 0.989 | Val acc: 0.882
Epoch 200 | Loss: 0.0741 | Train acc: 0.990 | Val acc: 0.884

GCN final train acc: 0.990
GCN final val acc  : 0.884
GCN final test acc : 0.869


## 10. Detailed comparison: per-class performance

Here we print a classification report for both:

- MLP (ANN)
- GCN (GNN)

This shows precision, recall and F1-score per class and overall.

Scikit-learn docs:  
https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html


In [12]:
y_cpu = y.cpu()
test_mask_cpu = test_mask.cpu()

with torch.no_grad():
    preds_mlp = logits_mlp.cpu().argmax(dim=-1)
    preds_gcn = logits_gcn.cpu().argmax(dim=-1)

print("=== MLP classification report (test) ===")
print(
    classification_report(
        y_cpu[test_mask_cpu],
        preds_mlp[test_mask_cpu],
        target_names=classes,
        digits=3,
    )
)

print("=== GCN classification report (test) ===")
print(
    classification_report(
        y_cpu[test_mask_cpu],
        preds_gcn[test_mask_cpu],
        target_names=classes,
        digits=3,
    )
)

print("\nSummary accuracies:")
print(f"MLP test accuracy: {test_acc_mlp:.3f}")
print(f"GCN test accuracy: {test_acc_gcn:.3f}")


=== MLP classification report (test) ===
                        precision    recall  f1-score   support

            Case_Based      0.750     0.700     0.724        60
    Genetic_Algorithms      0.804     0.892     0.846        83
       Neural_Networks      0.734     0.828     0.778       163
 Probabilistic_Methods      0.806     0.674     0.734        86
Reinforcement_Learning      0.750     0.628     0.684        43
         Rule_Learning      0.733     0.611     0.667        36
                Theory      0.625     0.634     0.629        71

              accuracy                          0.744       542
             macro avg      0.743     0.710     0.723       542
          weighted avg      0.745     0.744     0.741       542

=== GCN classification report (test) ===
                        precision    recall  f1-score   support

            Case_Based      0.926     0.833     0.877        60
    Genetic_Algorithms      0.929     0.952     0.940        83
       Neural_Netw